# EmpathRAG — Corpus Annotation (Day 10)
Annotate all 1,674,369 FAISS chunks with emotion labels using the trained RoBERTa checkpoint.  
**Runtime: A100 GPU. Expected time: ~25 minutes.**

This notebook:
1. Copies `metadata.db` from your local machine (uploaded here) to Colab
2. Loads the RoBERTa+LoRA checkpoint from Drive
3. Runs inference in batches of 512 on GPU
4. Writes emotion labels back to the SQLite db
5. Saves the annotated db back to Drive for you to download


In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"GPU  : {gpu}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
assert torch.cuda.is_available(), "Switch to A100 runtime."
print("✅ GPU ready.")


In [ ]:
!pip install -q peft>=0.18.0 accelerate>=1.0.0
import peft, accelerate
print(f"peft: {peft.__version__}  |  accelerate: {accelerate.__version__}")
print("✅ Packages ready.")


In [ ]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/empathrag/emotion_classifier"
DB_DRIVE_PATH  = "/content/drive/MyDrive/empathrag/metadata.db"
DB_LOCAL_PATH  = "/content/metadata.db"

# Verify checkpoint exists
required = ["adapter_config.json", "adapter_model.safetensors", "tokenizer.json"]
for f in required:
    path = os.path.join(CHECKPOINT_DIR, f)
    assert os.path.exists(path), f"Missing: {path}"
print(f"✅ Checkpoint found at: {CHECKPOINT_DIR}")


## ⬆️ Upload metadata.db

Run the cell below. It will prompt you to upload a file.  
Upload `data/indexes/metadata.db` from your local machine.

The file is ~1.26 GB — upload will take 2-5 minutes depending on your connection.


In [ ]:
from google.colab import files
import shutil, os

print("Click 'Choose Files' and select data/indexes/metadata.db from your local machine.")
print("File size: ~1.26 GB. Wait for upload to complete before proceeding.")

uploaded = files.upload()

# Move to working path
for fname in uploaded:
    dest = "/content/metadata.db"
    if fname != "metadata.db":
        os.rename(fname, dest)
    else:
        pass  # already at /content/metadata.db if uploaded directly

# Verify
import sqlite3
conn = sqlite3.connect(DB_LOCAL_PATH)
total      = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
annotated  = conn.execute("SELECT COUNT(*) FROM chunks WHERE emotion_label != -1").fetchone()[0]
unannotated= conn.execute("SELECT COUNT(*) FROM chunks WHERE emotion_label = -1").fetchone()[0]
conn.close()

print(f"\nDB loaded:")
print(f"  Total rows   : {total:,}")
print(f"  Annotated    : {annotated:,}  (already done — will skip)")
print(f"  Unannotated  : {unannotated:,}  (will process these)")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

LABEL_NAMES = ["distress", "anxiety", "frustration", "neutral", "hopeful"]
SAFETY_SCORE_MAP = {0: 0.0, 1: 0.0, 2: 0.3, 3: 0.7, 4: 1.0}
DEVICE = torch.device("cuda")

print("Loading RoBERTa + LoRA checkpoint onto GPU...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)
model = PeftModel.from_pretrained(base, CHECKPOINT_DIR).to(DEVICE).eval()

# Quick sanity check
enc = tokenizer("I feel completely overwhelmed and hopeless",
                return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
with torch.no_grad():
    pred = model(**enc).logits.argmax(-1).item()
print(f"Sanity check: 'overwhelmed and hopeless' → {LABEL_NAMES[pred]}")
assert LABEL_NAMES[pred] == "distress", f"Expected distress, got {LABEL_NAMES[pred]}"
print("✅ Model loaded and verified on GPU.")


In [ ]:
import sqlite3
from tqdm import tqdm

BATCH_SIZE = 512   # A100 can handle 512 at max_len=128 comfortably

conn = sqlite3.connect(DB_LOCAL_PATH)

# Only fetch rows not yet annotated — resumes safely if interrupted
rows = conn.execute(
    "SELECT id, text FROM chunks WHERE emotion_label = -1"
).fetchall()

print(f"Rows to annotate: {len(rows):,}")
if len(rows) == 0:
    print("Nothing to do — all rows already annotated.")
else:
    updates = []

    for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="Annotating"):
        batch  = rows[i : i + BATCH_SIZE]
        ids    = [r[0] for r in batch]
        texts  = [r[1] for r in batch]

        enc = tokenizer(
            texts,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**enc).logits

        labels = logits.argmax(-1).tolist()

        for rid, lbl in zip(ids, labels):
            score = SAFETY_SCORE_MAP[lbl]
            updates.append((lbl, score, rid))

        # Commit every 50 batches (~25,600 rows) to avoid losing work if disconnected
        if (i // BATCH_SIZE) % 50 == 0 and updates:
            conn.executemany(
                "UPDATE chunks SET emotion_label=?, safety_score=? WHERE id=?",
                updates
            )
            conn.commit()
            updates = []

    # Final commit
    if updates:
        conn.executemany(
            "UPDATE chunks SET emotion_label=?, safety_score=? WHERE id=?",
            updates
        )
        conn.commit()

print("\n✅ Annotation complete.")


In [ ]:
from collections import Counter

total      = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
annotated  = conn.execute("SELECT COUNT(*) FROM chunks WHERE emotion_label != -1").fetchone()[0]
unannotated= conn.execute("SELECT COUNT(*) FROM chunks WHERE emotion_label = -1").fetchone()[0]

print(f"Total rows   : {total:,}")
print(f"Annotated    : {annotated:,}")
print(f"Unannotated  : {unannotated:,}  ← must be 0")

rows = conn.execute("SELECT emotion_label FROM chunks").fetchall()
dist = Counter(r[0] for r in rows)
print("\nEmotion distribution:")
for i, name in enumerate(LABEL_NAMES):
    pct = 100 * dist[i] / total
    print(f"  {i} {name:<15} {dist[i]:>9,}  ({pct:.1f}%)")

conn.close()
assert unannotated == 0, f"{unannotated:,} rows still unannotated!"
print("\n✅ All rows annotated.")


In [ ]:
import shutil

print(f"Copying annotated DB to Drive: {DB_DRIVE_PATH}")
os.makedirs(os.path.dirname(DB_DRIVE_PATH), exist_ok=True)
shutil.copy2(DB_LOCAL_PATH, DB_DRIVE_PATH)

size_mb = os.path.getsize(DB_DRIVE_PATH) / 1e6
print(f"Saved: {DB_DRIVE_PATH}  ({size_mb:.1f} MB)")
print("\n✅ Done. Download metadata.db from Drive and replace data/indexes/metadata.db locally.")


## ✅ Annotation Complete — Download Steps

1. In Google Drive, navigate to `My Drive/empathrag/metadata.db`
2. Right-click → Download
3. Replace your local `data/indexes/metadata.db` with the downloaded file
4. Verify locally:
```python
import sqlite3
from collections import Counter
conn = sqlite3.connect('data/indexes/metadata.db')
unannotated = conn.execute('SELECT COUNT(*) FROM chunks WHERE emotion_label = -1').fetchone()[0]
print(f'Unannotated: {unannotated}')  # must be 0
conn.close()
```
